In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.utils.config import load_config, resolve_path

cfg = load_config()
tx = pd.read_parquet(resolve_path(cfg.get('paths.interim_data')))
print(f"{len(tx):,} cleaned transactions, {tx['customer_id'].nunique():,} customers")
tx.head()

In [ ]:
# Univariate: spend, frequency, recency distributions
feats = pd.read_parquet(resolve_path(cfg.get('paths.processed_features')))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['monetary_total', 'frequency', 'recency_days']):
    feats[col].clip(upper=feats[col].quantile(0.99)).hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
# Bivariate: features vs churn target
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['recency_days', 'frequency', 'monetary_total']):
    sns.boxplot(data=feats, x='churn', y=feats[col].clip(upper=feats[col].quantile(0.99)), ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
# Hypotheses the modeling stage will test (PRD Section 6.4):
# H1: recency_days is the dominant churn signal (strongest SHAP feature).
# H2: short-window momentum (spend_last_30d/90d) adds signal beyond RFM.
# H3: returns_ratio correlates positively with churn (dissatisfaction).
# H4: country effects exist but are weaker than behavioural features.
# H5: CLV is predictable for repeat customers, noisy for one-off buyers.

In [ ]:
# Multicollinearity check (VIF) before finalizing the feature set
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from src.models.train_churn import get_feature_columns
num_cols, _ = get_feature_columns(feats)
X = StandardScaler().fit_transform(feats[num_cols].astype(float))
vif = pd.DataFrame({'feature': num_cols,
    'vif': [variance_inflation_factor(X, i) for i in range(len(num_cols))]})
vif.sort_values('vif', ascending=False).head(10)